In [ ]:
import numpy as np
np.int = int; np.float = float

!pip install torch --index-url https://download.pytorch.org/whl/cpu -q
!pip install scipy pandas numba tqdm numpy biopython msgpack scikit-learn PyYAML -q

!rm -rf Enhanced-ECNet
!git clone https://github.com/itozheng-max/Enhanced-ECNet.git
%cd Enhanced-ECNet

!wget -q https://raw.githubusercontent.com/itozheng-max/Enhanced-ECNet/main/ecnet_rrm_data.tar.gz
!tar xzf ecnet_rrm_data.tar.gz
!ls data/RRM.braw data/distance_matrix.npy && echo 'Data OK' || echo 'Data MISSING'

import sys, time, torch
sys.path.insert(0, ".")
print(f"Py {sys.version.split()[0]} | Torch {torch.__version__} | GPU {torch.cuda.is_available()}")

In [ ]:
from ecnet.ecnet import ECNet
from ecnet.spatial_mask import SpatialMask

FASTA     = "data/RRM.fasta"
SINGLE_TSV = "data/RRM_single.tsv"
DOUBLE_TSV = "data/RRM_double.tsv"
BRAW      = "data/RRM.braw"
DIST_NPY  = "data/distance_matrix.npy"

print("OK")

In [ ]:
import ipywidgets as widgets
from IPython.display import display, Markdown
import pickle, os, time

kernel_select = widgets.Dropdown(options=['baseline','sigmoid','hill','surprise'], value='hill', description='Kernel:')
task_select   = widgets.Dropdown(options=[('Single->Single','ss'),('Single->Double','sd'),('Double->Double','dd')], value='sd', description='Task:')
seed_input    = widgets.IntText(value=42, description='Seed:')
d0_slider     = widgets.FloatSlider(value=8.0, min=4.0, max=14.0, step=0.5, description='d0 (A)')
gamma_slider  = widgets.FloatSlider(value=1.5, min=0.5, max=3.0, step=0.1, description='gamma')
n_slider      = widgets.IntSlider(value=4, min=1, max=10, step=1, description='n')
eps_slider    = widgets.FloatSlider(value=0.10, min=0.01, max=0.50, step=0.01, description='epsilon')
alpha_slider  = widgets.FloatSlider(value=1.0, min=0.0, max=5.0, step=0.5, description='alpha')
run_btn       = widgets.Button(description='Train & Save', button_style='success', icon='play')
save_name     = widgets.Text(value='my_experiment', description='Name:')
formula_out   = widgets.Output()
log_out       = widgets.Output()

RESULTS_DIR = './results'
os.makedirs(RESULTS_DIR, exist_ok=True)

def show_formula(kernel):
    f = {
        'baseline': r'$$ eij_{new} = eij $$',
        'sigmoid':  r'$$ W(d) = \frac{1}{1 + e^{\gamma (d - d_0)}} \qquad eij_{new} = \frac{eij}{W + \epsilon} $$',
        'hill':     r'$$ W(d) = \frac{1}{1 + (d/d_0)^n} \qquad eij_{new} = \frac{eij}{W + \epsilon} $$',
        'surprise': r'$$ W(d) = \frac{1}{1 + e^{\gamma (d - d_0)}} \qquad eij_{new} = \frac{eij}{W + \epsilon} \cdot \left[1 + \alpha(1-W)\tanh(|eij|_{norm})\right] $$',
    }
    formula_out.clear_output(wait=True)
    with formula_out:
        display(Markdown(f'### Kernel: {kernel}'))
        display(Markdown(f[kernel]))

def on_kernel_change(change):
    k = change['new']
    show_formula(k)
    d0_slider.layout.display    = 'none' if k == 'baseline' else ''
    gamma_slider.layout.display  = '' if k in ('sigmoid','surprise') else 'none'
    n_slider.layout.display      = '' if k == 'hill' else 'none'
    eps_slider.layout.display    = 'none' if k == 'baseline' else ''
    alpha_slider.layout.display  = '' if k == 'surprise' else 'none'

kernel_select.observe(on_kernel_change, 'value')
show_formula(kernel_select.value)
on_kernel_change({'new': kernel_select.value})

def build_mask():
    k = kernel_select.value
    if k == 'baseline': return None
    m = {'path': DIST_NPY, 'mode': 'surprise' if k == 'surprise' else 'divide',
         'd0': d0_slider.value, 'epsilon': eps_slider.value}
    if   k == 'sigmoid':  m.update({'kernel':'sigmoid', 'gamma':gamma_slider.value})
    elif k == 'hill':     m.update({'kernel':'hill',    'n':n_slider.value})
    elif k == 'surprise': m.update({'kernel':'sigmoid', 'gamma':gamma_slider.value, 'alpha':alpha_slider.value})
    return m

def run_one(name, mask):
    ts = task_select.value
    train_tsv = SINGLE_TSV if ts in ('ss','sd') else DOUBLE_TSV
    test_tsv  = SINGLE_TSV if ts == 'ss' else DOUBLE_TSV
    ecnet = ECNet(
        output_dir=f"./output/{name}", train_tsv=train_tsv, test_tsv=test_tsv,
        fasta=FASTA, ccmpred_output=BRAW, use_loc_feat=True, use_glob_feat=False,
        split_ratio=[0.9,0.1] if test_tsv else [0.7,0.1,0.2],
        random_seed=seed_input.value,
        n_ensembles=1, d_embed=20, d_model=64, d_h=64, batch_size=64,
        save_log=False, spatial_mask=mask)
    ecnet.train(epochs=200, patience=100, eval_freq=20, log_freq=200)
    r = ecnet.test(mode="ensemble", save_prediction=True)
    return r["metric"]["corr"], r["df"]

def on_run_clicked(b):
    run_btn.disabled = True
    log_out.clear_output(wait=True)
    with log_out:
        k = kernel_select.value
        task_names = {'ss':'Single->Single','sd':'Single->Double','dd':'Double->Double'}
        print(f'Task: {task_names[task_select.value]} | Kernel: {k} | Seed: {seed_input.value}')
        if k != 'baseline':
            parts = [f'd0={d0_slider.value}', f'eps={eps_slider.value}']
            if k in ('sigmoid','surprise'): parts.append(f'gamma={gamma_slider.value}')
            if k == 'hill': parts.append(f'n={n_slider.value}')
            if k == 'surprise': parts.append(f'alpha={alpha_slider.value}')
            print('  ' + ' '.join(parts))
        print()
        t0=time.time()
        bc, bdf = run_one('Baseline', None)
        print(f'  Baseline -> {bc:.6f}  ({time.time()-t0:.0f}s)')
        mask = build_mask()
        mc, mdf = None, None
        if mask is not None:
            t0=time.time()
            mc, mdf = run_one(k, mask)
            print(f'  {k:10s} -> {mc:.6f}  ({time.time()-t0:.0f}s)')
            print(f'\n  Delta: {mc-bc:+.6f} ({(mc/bc-1)*100:+.1f}%)')
        name = save_name.value or 'exp'
        ts_key = task_select.value
        ts = time.strftime('%Y%m%d_%H%M%S')
        fname = f'{RESULTS_DIR}/{name}_{ts_key}_{ts}.pkl'
        data = {'task':ts_key, 'kernel':k, 'seed':seed_input.value,
                'baseline_corr':bc, 'mask_corr':mc,
                'delta':(mc-bc) if mc is not None else None,
                'params':{'d0':d0_slider.value if k!='baseline' else None,
                          'epsilon':eps_slider.value if k!='baseline' else None,
                          'gamma':gamma_slider.value if k in ('sigmoid','surprise') else None,
                          'n':n_slider.value if k=='hill' else None,
                          'alpha':alpha_slider.value if k=='surprise' else None},
                'baseline_df':bdf, 'mask_df':mdf}
        with open(fname,'wb') as f: pickle.dump(data, f)
        print(f'\n  Saved -> {fname}')
    run_btn.disabled = False

run_btn.on_click(on_run_clicked)

display(Markdown('### Task & Seed'))
display(widgets.HBox([task_select, seed_input]))
display(Markdown('---'))
display(Markdown('### Kernel & Parameters'))
display(kernel_select)
display(formula_out)
display(d0_slider)
display(gamma_slider)
display(n_slider)
display(eps_slider)
display(alpha_slider)
display(Markdown('---'))
display(widgets.HBox([save_name, run_btn]))
display(Markdown('---'))
display(log_out)